# logit-thermo A2-5b：T_0 での f の傾き（CPU）

- **必要な secret：なし**。**ランタイム：CPU（標準）**。見込みは 10〜15 分。
- ゲート：`docs/A2_5B_GATE.md`。A2-5 の 9 回（`A2_5/*.npz`）から、s ＝（F(0.9) − F(0.8)）÷ 0.1 を出し、Q5〜Q7 を判定して `A2_5/a2_5b_summary.json` に書きます。**結果は表示しません。**
- 計算のコード：`scripts/a2_2_analyze.py`・`scripts/a2_5_analyze.py` は Drive の前のノートブックから取り出してハッシュで確かめ、`scripts/a2_5b_slope.py`（`0a1f9548ecb8906e…`）は埋め込み（データを開く前に固定）。記事の本文は使いません。
- **通知**：🚀・🎉・⚠️。名札は「logit-thermo A2-5b 分析（T_0 での f の傾き、CPU）」。終わったら、またはエラーで止まったら、ランタイムを切ります。


In [ ]:
# セル1：通知、GPU、Drive
import os, time, urllib.request
NTFY_TOPIC = 'YOUR_NTFY_TOPIC'
NB_LABEL = 'logit-thermo A2-5b 分析（T_0 での f の傾き、CPU）'
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=(msg + '\n[' + NB_LABEL + ']').encode('utf-8'),
                                     headers={'Title': 'logit-thermo - A2-5b slope', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        import traceback
        e = result.error_in_exec
        txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
        v = os.environ.get('HF_TOKEN')
        if v: txt = txt.replace(v, '***')
        txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
        notify('⚠️ 止まりました（エラー）：\n' + txt, tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# セル2：分析のコード
import ast, glob, hashlib, json
def take_src(want):
    """want：{変数の名前: sha256}。Drive の logit-thermo のノートブックから、埋め込んだコードを取り出してハッシュで確かめる。"""
    got = {}
    for p in sorted(glob.glob('/content/drive/MyDrive/logit-thermo' + '/**/*.ipynb', recursive=True)):
        try:
            cells = json.load(open(p))['cells']
        except Exception:
            continue
        for c in cells:
            s = c['source'] if isinstance(c['source'], str) else ''.join(c['source'])
            if c.get('cell_type') != 'code' or 'SRC' not in s:
                continue
            try:
                body = ast.parse(s).body
            except SyntaxError:
                continue
            for n in body:
                if isinstance(n, ast.Assign) and isinstance(n.targets[0], ast.Name) and n.targets[0].id in want and n.targets[0].id not in got:
                    v = ast.literal_eval(n.value)
                    if hashlib.sha256(v.encode()).hexdigest() == want[n.targets[0].id]:
                        got[n.targets[0].id] = v
        if len(got) == len(want):
            break
    miss = [k for k in want if k not in got]
    assert not miss, ('Drive のノートブックからコードを取り出せませんでした（前のノートブックを動かしていませんか）：', miss)
    return got
_s = take_src({'SRCA22': 'fe1f96db1f7e84031e85a70bcc8dcbb95c9fc06b3367657ff4074a9f9d1343f7', 'SRC5A': 'f4ef35b9f594a205ac1145c1a326e9de8a25280171fe2b93083cc1676e58d3ae'})
SRCA22, SRC5A = _s['SRCA22'], _s['SRC5A']
SRC5B = '"""A2-5b（ゲート docs/A2_5B_GATE.md）：T_0 での f の傾き。データを開く前に固定する。CPU。\nscripts/a2_2_analyze.py（per_doc_feedback）と scripts/a2_5_analyze.py（axis_word・A5_SIZE・A5_STEP）を実行したのと\n同じ名前空間で実行する。記事の本文はいらない（npz の tf と gen だけ使う）。\n\n    analyze5b(a2_5 のフォルダ) -> dict\n"""\nimport math\nimport numpy as np\n\nB5_T = (0.8, 0.9)                      # 傾きをとる 2 つの温度\nB5_BOOT, B5_SEED = 1000, 20261103      # 記事を引き直すブートストラップ（全モデルで同じ引き方＝対応あり）\nB5_Q7 = (1.5, 3.0)                     # 70M ÷ 2.8B の線\n\n\ndef slope_of(fd, temps, idx=None):\n    """fd：(nD, nT) の記事ごとの f。→ s ＝（F(0.9) − F(0.8)）÷ 0.1。F は記事の中央値（T_0 と同じ量）。"""\n    k0, k1 = temps.index(B5_T[0]), temps.index(B5_T[1])\n    x = fd if idx is None else fd[idx]\n    F = np.median(x, 0)\n    return float((F[k1] - F[k0]) / (temps[k1] - temps[k0]))\n\n\ndef q7_word(r):\n    if not np.isfinite(r):\n        return \'なし\'\n    return \'1.5 倍より小さい\' if r < B5_Q7[0] else \'1.5〜3 倍\' if r <= B5_Q7[1] else \'3 倍より大きい\'\n\n\ndef analyze5b_from_fd(fds, temps):\n    """fds：{名前: (nD, nT)}。全モデルで記事の並びが同じであること。"""\n    nD = {v.shape[0] for v in fds.values()}\n    assert len(nD) == 1, \'記事の数がモデルによって違います\'\n    nD = nD.pop()\n    rng = np.random.default_rng(B5_SEED)\n    idx = rng.integers(0, nD, size=(B5_BOOT, nD))\n    s = {n: slope_of(fd, temps) for n, fd in fds.items()}\n    sb = {n: np.array([slope_of(fd, temps, idx[b]) for b in range(B5_BOOT)]) for n, fd in fds.items()}\n    ci = {n: [float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))] for n, v in sb.items()}\n    res = {\'s\': s, \'s_ci\': ci, \'T\': list(B5_T), \'boot\': B5_BOOT, \'seed\': B5_SEED}\n    size = [n for n, _ in A5_SIZE]; step = [n for n, _ in A5_STEP]\n    xs = [math.log(p) for _, p in A5_SIZE]\n    sym, det = axis_word(xs, [s[n] for n in size], np.stack([sb[n] for n in size], 1), True, False, \'+\', \'-\')\n    res[\'Q5\'] = {\'＋\': \'大きいほど急\', \'−\': \'小さいほど急\'}.get(sym, \'はっきりしない\'); res[\'Q5_detail\'] = det\n    xt = [math.log(t) for _, t in A5_STEP]\n    sym, det = axis_word(xt, [s[n] for n in step], np.stack([sb[n] for n in step], 1), True, True, \'+\', \'-\')\n    res[\'Q6\'] = {\'＋\': \'学ぶほど急\', \'−\': \'学ぶほど緩やか\'}.get(sym, \'はっきりしない\'); res[\'Q6_detail\'] = det\n    r = s[\'pythia-70m\'] / s[\'pythia-2.8b\'] if s[\'pythia-2.8b\'] != 0 else float(\'nan\')\n    rb = sb[\'pythia-70m\'] / sb[\'pythia-2.8b\']\n    res[\'Q7\'] = q7_word(r)\n    res[\'Q7_detail\'] = {\'ratio\': r, \'ci\': [float(np.nanpercentile(rb, 2.5)), float(np.nanpercentile(rb, 97.5))]}\n    # 記録（判定しない）：ΔH 相当（T_0 × s）は T_0 を別に読むので、ここでは s だけ残す\n    return res\n\n\ndef analyze5b(d5):\n    names = [n for n, _ in A5_SIZE] + [n for n, _ in A5_STEP if n not in dict(A5_SIZE)]\n    fds, temps, uses = {}, None, set()\n    for n in names:\n        Z = np.load(f\'{d5}/{n}.npz\')\n        t = [float(x) for x in Z[\'temps\']]\n        assert temps is None or t == temps, \'温度の並びが違います\'\n        temps = t\n        uses.add(tuple(int(u) for u in Z[\'use\']))\n        fds[n] = per_doc_feedback(Z[\'tf\'][0], Z[\'gen\'][0])\n    assert len(uses) == 1, \'使った記事がモデルによって違います\'\n    return analyze5b_from_fd(fds, temps)\n\n\ndef self_test_a25b():\n    temps = [0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 1.0]\n    rng = np.random.default_rng(1)\n    def fake(k):  # 傾き k（0.8→0.9 で 0.1k 上がる）＋小さな雑音\n        base = np.array([-3, -1, 0, 0.5, 1, 1.5, 2.0])\n        base[4] = base[2] + 0.1 * k\n        return base[None, :] + rng.normal(0, 0.01, (40, len(temps)))\n    sizes = [n for n, _ in A5_SIZE]; steps = [n for n, _ in A5_STEP]\n    ks = dict(zip(sizes, [12, 10, 8, 6, 4, 2]))\n    ks.update(zip(steps, [2, 3, 4, 6]))         # pythia-1b は両方の軸にいる：大きさの軸で 6、学びの最後でも 6\n    fds = {n: fake(k) for n, k in ks.items()}\n    r = analyze5b_from_fd(fds, temps)\n    assert abs(r[\'s\'][\'pythia-70m\'] - 12) < 0.5, r[\'s\']\n    assert r[\'Q5\'] == \'小さいほど急\', r\n    assert r[\'Q6\'] == \'学ぶほど急\', r\n    assert r[\'Q7\'] == \'3 倍より大きい\', r\n    assert q7_word(1.49) == \'1.5 倍より小さい\' and q7_word(3.0) == \'1.5〜3 倍\' and q7_word(float(\'nan\')) == \'なし\'\n    return \'OK\'\n'
assert hashlib.sha256(SRC5B.encode()).hexdigest() == '0a1f9548ecb8906e00e481e71e1cd13bebc76ad93abe902b5edd54b8b1bea4ef'
exec(SRCA22)
exec(SRC5A)
exec(SRC5B)
print('自己テスト：', self_test_a25b())


In [ ]:
# セル3：分析して保存（結果は表示しない）
import json
try:
    r = analyze5b('/content/drive/MyDrive/logit-thermo/A2_5')
    r['a2_5b_slope_sha256'] = '0a1f9548ecb8906e00e481e71e1cd13bebc76ad93abe902b5edd54b8b1bea4ef'
    json.dump(_clean(r), open('/content/drive/MyDrive/logit-thermo/A2_5' + '/a2_5b_summary.json', 'w'), ensure_ascii=False, separators=(',', ':'))
    print('保存しました')
except Exception as e:
    import traceback
    _NOTIFIED['error'] = True
    txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
    v = os.environ.get('HF_TOKEN')
    if v: txt = txt.replace(v, '***')
    txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
    notify('⚠️ A2-5b の分析で止まりました。ランタイムを切ります：\n' + txt, tags='warning')
    if globals().get('AUTO_DISCONNECT', True):
        from google.colab import drive as _d, runtime as _r
        _d.flush_and_unmount(); time.sleep(3); _r.unassign()
    raise


In [ ]:
# 最後：終わりの通知と、ランタイムを切る
notify('🎉 終わりました：判定と要約を A2_5/a2_5b_summary.json に保存しました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount(); time.sleep(3); _runtime.unassign()
